# Lab 6 — Dashboard Comparativo LangFuse
## Análise Final: Custo, Latência e Qualidade das Técnicas da Aula 11

**Aula 11 · MBA RAG & CAG Aplicados a Direito e Segurança Pública**

**Objetivo:** Consolidar métricas de todos os labs em um dashboard comparativo, analisando custo de tokens com/sem LLMLingua, latência ColBERT vs BGE-M3, e qualidade de resposta com/sem DSPy.

**Duração estimada:** 30 minutos

---

### O que mede o LangFuse?

```
LangFuse = Observabilidade para pipelines LLM

Captura:
  • Traces: cada chamada end-to-end ao pipeline
  • Spans: etapas individuais (retrieve, compress, generate)
  • Tokens: input, output, custo por modelo
  • Latência: tempo por etapa
  • Scores: métricas de qualidade (Faithfulness, etc.)
  • Metadados: query, contexto, resposta
```

## Etapa 0 — Instalação

In [ ]:
!pip install -q langfuse==2.50.0
!pip install -q matplotlib==3.9.0
!pip install -q pandas==2.2.2

print("✅ Dependências instaladas!")

## Etapa 1 — Configuração do LangFuse

Para usar o LangFuse cloud: crie conta gratuita em https://cloud.langfuse.com e configure as chaves abaixo.

In [ ]:
import os

# =============================================================
# OPÇÃO 1: LangFuse Cloud (gratuito até 50k observações/mês)
# =============================================================
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-XXXX"
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-XXXX"
# os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

# from langfuse import Langfuse
# langfuse = Langfuse()

# =============================================================
# OPÇÃO 2: Mock LangFuse para demonstração (sem API)
# =============================================================
print("⚠️  Modo simulação ativo (Mock LangFuse)")
print("   Para usar LangFuse real: configure chaves na OPÇÃO 1")
print("   Criar conta: https://cloud.langfuse.com (gratuito)")
print()


class MockLangfuseSpan:
    def __init__(self, name, traces):
        self.name = name
        self.traces = traces
    def end(self, **kwargs): self.traces.append({"span": self.name, **kwargs})
    def score(self, **kwargs): self.traces.append({"score": self.name, **kwargs})
    def __enter__(self): return self
    def __exit__(self, *args): self.end()

class MockLangfuse:
    def __init__(self):
        self.traces = []
        self.observations = []
    def trace(self, name, **kwargs):
        trace = {"name": name, "spans": [], **kwargs}
        self.traces.append(trace)
        return MockLangfuseSpan(name, self.observations)
    def span(self, name, **kwargs): return MockLangfuseSpan(name, self.observations)
    def flush(self): pass

langfuse = MockLangfuse()
print("✅ LangFuse (modo simulação) inicializado")

## Etapa 2 — Dados de Métricas dos Labs (Simulados)

Em produção, esses dados viriam automaticamente do LangFuse após executar os Labs 1-5 instrumentados.

In [ ]:
import pandas as pd
import numpy as np

# Dados simulados representando execuções reais dos Labs 1-5
# Em produção: exportar do LangFuse via API ou dashboard

# ============================================================
# Lab 2: Comparativo de custo com/sem LLMLingua
# ============================================================
dados_llmlingua = pd.DataFrame({
    "configuracao": ["Sem LLMLingua", "LLMLingua 50%", "LLMLingua 70%", "LLMLingua 85%"],
    "tokens_contexto_medio": [512, 256, 154, 77],
    "tokens_prompt_total": [620, 364, 262, 185],
    "custo_por_query_usd": [0.00310, 0.00182, 0.00131, 0.00093],
    "faithfulness_media": [0.91, 0.89, 0.85, 0.71],
    "latencia_llm_ms": [1850, 1120, 820, 610],
    "latencia_compressor_ms": [0, 210, 210, 210],
    "latencia_total_ms": [1850, 1330, 1030, 820],
})

# ============================================================
# Lab 3: Comparativo de retrieval ColBERT vs BGE-M3
# ============================================================
dados_retrieval = pd.DataFrame({
    "metodo": ["BM25 (baseline)", "BGE-M3 (bi-encoder)", "ColBERT (late interaction)"],
    "ndcg_10_medio": [0.48, 0.67, 0.79],
    "latencia_busca_ms": [5, 18, 45],
    "tamanho_indice_mb": [12, 48, 185],
    "custo_indexacao_min": [0.5, 3.0, 12.0],
})

# ============================================================
# Lab 5: Comparativo de qualidade Manual vs DSPy
# ============================================================
dados_dspy = pd.DataFrame({
    "pipeline": ["RAG Manual", "RAG + DSPy (BootstrapFewShot)", "RAG + DSPy (MIPROv2)"],
    "faithfulness_media": [0.62, 0.81, 0.87],
    "answer_accuracy": [0.55, 0.74, 0.81],
    "artigos_citados_pct": [0.34, 0.89, 0.93],
    "tokens_prompt_medio": [380, 890, 1020],
    "custo_compilacao_usd": [0.00, 0.85, 4.20],
})

# ============================================================
# Visão consolidada: todas as técnicas
# ============================================================
dados_pipeline_completo = pd.DataFrame({
    "pipeline": [
        "RAG Básico (BM25 + Sem compress.)",
        "RAG Melhorado (BGE-M3 + Sem compress.)",
        "RAG + LLMLingua 70% (BGE-M3)",
        "RAG + ColBERT (Sem compress.)",
        "RAG + ColBERT + LLMLingua 70%",
        "RAG + ColBERT + LLMLingua + DSPy",
    ],
    "ndcg_10": [0.48, 0.67, 0.67, 0.79, 0.79, 0.79],
    "faithfulness": [0.55, 0.68, 0.65, 0.72, 0.69, 0.87],
    "custo_por_1k_queries_usd": [3.10, 3.10, 1.31, 3.10, 1.31, 1.50],
    "latencia_total_ms": [1860, 1870, 1050, 1915, 1095, 1200],
})

print("📊 Dados carregados:")
print(f"  Lab 2 (LLMLingua): {len(dados_llmlingua)} configurações")
print(f"  Lab 3 (Retrieval): {len(dados_retrieval)} métodos")
print(f"  Lab 5 (DSPy):     {len(dados_dspy)} pipelines")
print(f"  Consolidado:       {len(dados_pipeline_completo)} combinações")

## Etapa 3 — Dashboard de Visualizações

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configuração visual
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'Dashboard — Aula 11: Análise Comparativa das Técnicas Complementares\n'
    'MBA RAG & CAG Aplicados a Direito e Segurança Pública',
    fontsize=13, fontweight='bold', y=0.98
)

CORES = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6', '#1ABC9C']

# ============================================================
# GRÁFICO 1: LLMLingua — Custo vs Faithfulness
# ============================================================
ax1 = axes[0, 0]
x = np.arange(len(dados_llmlingua))
bars = ax1.bar(x, dados_llmlingua['custo_por_query_usd'] * 1000,
               color=CORES[:4], alpha=0.85, zorder=3)

ax1b = ax1.twinx()
ax1b.plot(x, dados_llmlingua['faithfulness_media'],
          'o--', color='#E74C3C', linewidth=2, markersize=8, zorder=4)
ax1b.set_ylim(0.5, 1.0)
ax1b.set_ylabel('Faithfulness', color='#E74C3C', fontsize=9)
ax1b.tick_params(axis='y', labelcolor='#E74C3C')

ax1.set_xticks(x)
ax1.set_xticklabels(dados_llmlingua['configuracao'], rotation=20, ha='right', fontsize=8)
ax1.set_ylabel('Custo (mUSD / query)', fontsize=9)
ax1.set_title('LLMLingua: Trade-off Custo × Faithfulness', fontsize=10, pad=10)
ax1.grid(axis='y', alpha=0.3, zorder=0)

for bar, val in zip(bars, dados_llmlingua['custo_por_query_usd'] * 1000):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', va='bottom', fontsize=8)

# ============================================================
# GRÁFICO 2: Retrieval — nDCG@10 vs Latência
# ============================================================
ax2 = axes[0, 1]
scatter = ax2.scatter(
    dados_retrieval['latencia_busca_ms'],
    dados_retrieval['ndcg_10_medio'],
    s=dados_retrieval['tamanho_indice_mb'] * 2,
    c=CORES[:3], alpha=0.85, zorder=3
)

for i, row in dados_retrieval.iterrows():
    ax2.annotate(
        row['metodo'],
        (row['latencia_busca_ms'], row['ndcg_10_medio']),
        textcoords='offset points', xytext=(8, 5),
        fontsize=8, fontweight='bold'
    )

ax2.set_xlabel('Latência de Busca (ms)', fontsize=9)
ax2.set_ylabel('nDCG@10', fontsize=9)
ax2.set_title('Retrieval: nDCG@10 × Latência\n(tamanho = tamanho do índice)', fontsize=10, pad=10)
ax2.grid(alpha=0.3)
ax2.set_xlim(-5, 60)
ax2.set_ylim(0.3, 0.9)

# Quadrante ideal
ax2.axhline(0.67, color='gray', linestyle=':', alpha=0.5)
ax2.text(50, 0.68, 'baseline\nBGE-M3', fontsize=7, color='gray', ha='right')

# ============================================================
# GRÁFICO 3: DSPy — Qualidade Antes/Depois
# ============================================================
ax3 = axes[1, 0]
metricas = ['faithfulness_media', 'answer_accuracy', 'artigos_citados_pct']
labels_metricas = ['Faithfulness', 'Answer Accuracy', '% com Artigo Citado']
x = np.arange(len(metricas))
width = 0.25

for i, (_, row) in enumerate(dados_dspy.iterrows()):
    valores = [row[m] for m in metricas]
    bars = ax3.bar(x + i * width, valores, width, label=row['pipeline'],
                   color=CORES[i], alpha=0.85, zorder=3)

ax3.set_xticks(x + width)
ax3.set_xticklabels(labels_metricas, fontsize=9)
ax3.set_ylabel('Score (0.0 – 1.0)', fontsize=9)
ax3.set_title('DSPy: Qualidade das Respostas por Pipeline', fontsize=10, pad=10)
ax3.legend(fontsize=7, loc='lower right')
ax3.set_ylim(0, 1.1)
ax3.grid(axis='y', alpha=0.3, zorder=0)
ax3.axhline(0.8, color='green', linestyle='--', alpha=0.4, label='Meta 80%')
ax3.text(2.65, 0.81, 'meta 80%', fontsize=7, color='green', alpha=0.7)

# ============================================================
# GRÁFICO 4: Pipeline Completo — Comparativo Final
# ============================================================
ax4 = axes[1, 1]

# Normalizar métricas para radar-style bar chart
pipelines = dados_pipeline_completo['pipeline'].str[:35]
ndcg_norm = dados_pipeline_completo['ndcg_10'] / dados_pipeline_completo['ndcg_10'].max()
faith_norm = dados_pipeline_completo['faithfulness'] / dados_pipeline_completo['faithfulness'].max()
custo_norm = 1 - (dados_pipeline_completo['custo_por_1k_queries_usd'] / dados_pipeline_completo['custo_por_1k_queries_usd'].max())
lat_norm = 1 - (dados_pipeline_completo['latencia_total_ms'] / dados_pipeline_completo['latencia_total_ms'].max())

score_geral = (ndcg_norm * 0.35 + faith_norm * 0.35 + custo_norm * 0.15 + lat_norm * 0.15)

y_pos = np.arange(len(pipelines))
barras = ax4.barh(y_pos, score_geral, color=CORES[:len(pipelines)], alpha=0.85, zorder=3)

ax4.set_yticks(y_pos)
ax4.set_yticklabels(pipelines, fontsize=8)
ax4.set_xlabel('Score Geral Normalizado', fontsize=9)
ax4.set_title('Ranking Final: Score Ponderado\n(35% nDCG + 35% Faithful + 15% Custo + 15% Latência)',
              fontsize=10, pad=10)
ax4.grid(axis='x', alpha=0.3, zorder=0)
ax4.set_xlim(0, 1.1)

for bar, val in zip(barras, score_geral):
    ax4.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=8, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('/content/dashboard_aula11.png', bbox_inches='tight', dpi=150)
plt.show()
print("\n✅ Dashboard salvo em /content/dashboard_aula11.png")

## Etapa 4 — Tabela Consolidada de Decisão

In [ ]:
# Tabela de decisão para o projeto final

print("=" * 80)
print("GUIA DE DECISÃO — QUAL TÉCNICA USAR NO PROJETO FINAL?")
print("MBA RAG & CAG — Aula 11")
print("=" * 80)

tabela_decisao = [
    {
        "Técnica": "Multimodal RAG",
        "Use se...": "Corpus tem imagens, tabelas, diagramas",
        "Evite se...": "Corpus é só texto; custo de infra é crítico",
        "Custo relativo": "Alto",
        "Complexidade": "Alta",
        "Ganho qualidade": "Alto (em docs mistos)"
    },
    {
        "Técnica": "LLMLingua (70%)",
        "Use se...": "Custo de tokens é crítico; contexto grande",
        "Evite se...": "Textos com exceções sutis; decisões críticas",
        "Custo relativo": "Reduz 55%",
        "Complexidade": "Baixa",
        "Ganho qualidade": "Neutro (-6% faith.)"
    },
    {
        "Técnica": "ColBERT",
        "Use se...": "Alta precisão em queries técnicas específicas",
        "Evite se...": "Hardware limitado; corpus < 10k docs",
        "Custo relativo": "Médio-Alto",
        "Complexidade": "Média",
        "Ganho qualidade": "Alto (+18% nDCG)"
    },
    {
        "Técnica": "Time-Aware RAG",
        "Use se...": "Corpus tem leis/regulamentos com validade temporal",
        "Evite se...": "Corpus estático e homogêneo no tempo",
        "Custo relativo": "Baixo",
        "Complexidade": "Baixa",
        "Ganho qualidade": "Alto (evita docs revogados)"
    },
    {
        "Técnica": "DSPy",
        "Use se...": "Pipeline existe; quer melhoria sistemática",
        "Evite se...": "Sem dataset avaliação; orçamento de compute restrito",
        "Custo relativo": "Médio (compilação)",
        "Complexidade": "Alta",
        "Ganho qualidade": "Alto (+30% faith.)"
    },
]

df_decisao = pd.DataFrame(tabela_decisao)
print(df_decisao.to_string(index=False))

print("""
==========================================================================
RECOMENDAÇÃO PARA PROJETO FINAL (Domínio Jurídico/Segurança Pública)
==========================================================================

Pipeline Mínimo Viável (MVP):
  BGE-M3 + Hybrid Search (Aula 5) + Time-Aware RAG
  → Cobre: relevância semântica + lexical + temporal

Pipeline Avançado (produção):
  BGE-M3 + Hybrid Search + Time-Aware RAG
  + ColBERT Reranker (para precisão)
  + LLMLingua 70% (para custo)
  + DSPy (otimização contínua)
  → Cobre todas as dimensões com trade-off otimizado

Quando adicionar Multimodal:
  Apenas se corpus inclui laudos periciais com imagens,
  ou documentos de inteligência com diagramas e mapas.
""")

## Etapa 5 — Integração com LangFuse Real

In [ ]:
# Código de integração LangFuse para instrumentar o pipeline
# REFERÊNCIA: Executar em produção com cliente real configurado

CODIGO_LANGFUSE = '''
from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context

langfuse = Langfuse(
    public_key="pk-lf-XXX",
    secret_key="sk-lf-XXX",
    host="https://cloud.langfuse.com"
)


@observe(name="pipeline_rag_juridico")
def pipeline_rag_instrumentado(
    query: str,
    usar_colbert: bool = True,
    usar_llmlingua: bool = True
) -> dict:
    """Pipeline RAG com observabilidade LangFuse automática."""
    
    # Etapa 1: Retrieve (instrumentado automaticamente pelo @observe)
    with langfuse.span(name="retrieve") as span:
        docs = colbert_search(query, k=5) if usar_colbert else bge_search(query, k=5)
        span.end(metadata={"n_docs": len(docs), "metodo": "colbert" if usar_colbert else "bge"})
    
    # Etapa 2: Comprimir (se habilitado)
    if usar_llmlingua:
        with langfuse.span(name="compress") as span:
            contexto_comprimido, tokens_antes, tokens_depois = comprimir(docs, query)
            span.end(metadata={"tokens_antes": tokens_antes, "tokens_depois": tokens_depois})
    else:
        contexto_comprimido = "\\n".join(docs)
    
    # Etapa 3: Gerar resposta
    with langfuse.span(name="generate") as span:
        resposta, usage = gerar_resposta(contexto_comprimido, query)
        span.end(
            usage={"input": usage.prompt_tokens, "output": usage.completion_tokens},
            model="gpt-4o-mini"
        )
    
    # Etapa 4: Avaliar com RAGAS
    faithfulness_score = avaliar_faithfulness(resposta, docs)
    langfuse_context.score_current_trace(
        name="faithfulness",
        value=faithfulness_score
    )
    
    return {"resposta": resposta, "faithfulness": faithfulness_score}
'''

print("📋 Código de integração LangFuse:")
print(CODIGO_LANGFUSE)

## Exercício Final Integrador

> **Exercício Integrador:** Com base nos dados do dashboard, escreva um relatório técnico de 1 página (em Markdown) recomendando a arquitetura ideal para um sistema RAG a ser implantado em uma delegacia especializada em crimes contra o patrimônio. O relatório deve:
> 1. Justificar a escolha de cada técnica com dados do dashboard
> 2. Estimar o custo mensal para 10.000 consultas/dia
> 3. Identificar os dois principais riscos técnicos e como mitigá-los
> 4. Propor um plano de monitoramento com LangFuse

## Pergunta de Reflexão Final

> **Reflexão:** Ao longo das Aulas 1-11, você aprendeu 25+ técnicas de RAG. Qual seria o pipeline "suficientemente bom" (good enough) para um sistema de assessoria jurídica real, considerando que:
> - Erros têm consequências jurídicas graves
> - O orçamento da delegacia é limitado
> - A equipe técnica tem 2 desenvolvedores
> - O corpus tem 50.000 documentos (legislação + jurisprudência)
> 
> Defenda sua escolha com argumentos técnicos e econômicos.

---

## Resumo da Aula 11

| Técnica | Lab | Ganho Principal | Complexidade |
|---|---|---|---|
| Multimodal RAG | Lab 1 | Recupera texto + imagem | Alta |
| LLMLingua | Lab 2 | -55% custo de tokens | Baixa |
| ColBERT | Lab 3 | +18% nDCG@10 | Média |
| Time-Aware | Lab 4 | Evita docs revogados | Baixa |
| DSPy | Lab 5 | +30% Faithfulness | Alta |
| LangFuse | Lab 6 | Observabilidade contínua | Baixa |

**Próxima:** Aula 12 — Projeto Final: Arquitetura RAG de Produção